# Exploring Ibis with iacs Data

Ibis provides a backend-agnostic dataframe API that compiles to SQL.
Here we use the DuckDB backend to explore iacs component data.

Key idea: Ibis expressions are **lazy** — nothing executes until you
call `.to_pandas()` or `.execute()`.

In [1]:
import ibis
import pandas as pd

from iacs.io_system import IOSystem
from iacs.registry import Registry

ibis.options.interactive = True  # auto-display results (like pandas)

## 1. Load iacs data into a Registry

In [2]:
io = IOSystem()
entity_centered = io.read_entity_centered_file("../components/components.yaml")
registry = Registry.from_entity_centered(entity_centered)

print(f"Component types: {registry.component_types}")

Component types: ['description', 'file_info', 'id', 'input', 'output', 'parent', 'requirement', 'schema', 'system', 'todo']


## 2. Connect to DuckDB via Ibis and register tables

In [3]:
con = ibis.duckdb.connect()

# Register each component table
tables = {}
for comp_type in registry.component_types:
    df = registry.view(comp_type).reset_index()
    table_name = comp_type.replace(" ", "_")
    try:
        con.create_table(table_name, df)
        tables[table_name] = con.table(table_name)
        print(f"Registered '{table_name}' ({len(df)} rows)")
    except Exception as e:
        print(f"Error registering '{table_name}': {e}")

Registered 'description' (123 rows)
Registered 'file_info' (1 rows)
Registered 'id' (145 rows)
Registered 'input' (8 rows)
Registered 'output' (7 rows)
Registered 'parent' (154 rows)
Error registering 'requirement': ("Expected bytes, got a 'dict' object", 'Conversion failed for column value with type object')
Error registering 'schema': ("Expected bytes, got a 'dict' object", 'Conversion failed for column value with type object')
Registered 'system' (4 rows)
Registered 'todo' (16 rows)


## 3. Basic operations: select, filter, limit

In [4]:
# Look at the description table
description = tables["description"]
description

┏━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ entity_id    ┃ component_index ┃ component_type ┃ value                                                                            ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ string       │ int64           │ string         │ string                                                                           │
├──────────────┼─────────────────┼────────────────┼──────────────────────────────────────────────────────────────────────────────────┤
│ 080505fba0c9 │               1 │ description    │ This file contains the key entities that define and describe how iacs works.\n   │
│ 085b2f4e818d │               2 │ description    │ iacs (short for Infrastructure-as-Code Sketch) is software designed to help tec… │
│ 847469f00f2d │               2 │ description    │ Provide a means to comprehensively and accurately document infrastructure.\n     │
│ 5cdb11e7f0af │               2 │ description    │ If infrastructure is defined elsewhere in an existing format, we don't want to … │
│ aa4b968e2d6d │               2 │ description    │ We don't want to duplicate other IaC code if possible.\n                         │
│ 669b4507e408 │               2 │ description    │ Code such as Python also provides clear definitions.\n                           │
│ 4834288e3da8 │               2 │ description    │ Version control is essential for reliability.\n                                  │
│ 0d2717c66a9f │               2 │ description    │ A source of truth is not *a* source of truth if it is distributed.\n             │
│ bc4092fa60fb │               2 │ description    │ This is one of the key features iacs provides that other modeling tools do not-… │
│ bc4f5d47c6ec │               2 │ description    │ Users need to be able to describe the infrastructure they want to build.\n       │
│ …            │               … │ …              │ …                                                                                │
└──────────────┴─────────────────┴────────────────┴──────────────────────────────────────────────────────────────────────────────────┘

In [5]:
# Select specific columns
description.select("entity_id", "value").limit(10)

┏━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ entity_id    ┃ value                                                                            ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ string       │ string                                                                           │
├──────────────┼──────────────────────────────────────────────────────────────────────────────────┤
│ 080505fba0c9 │ This file contains the key entities that define and describe how iacs works.\n   │
│ 085b2f4e818d │ iacs (short for Infrastructure-as-Code Sketch) is software designed to help tec… │
│ 847469f00f2d │ Provide a means to comprehensively and accurately document infrastructure.\n     │
│ 5cdb11e7f0af │ If infrastructure is defined elsewhere in an existing format, we don't want to … │
│ aa4b968e2d6d │ We don't want to duplicate other IaC code if possible.\n                         │
│ 669b4507e408 │ Code such as Python also provides clear definitions.\n                           │
│ 4834288e3da8 │ Version control is essential for reliability.\n                                  │
│ 0d2717c66a9f │ A source of truth is not *a* source of truth if it is distributed.\n             │
│ bc4092fa60fb │ This is one of the key features iacs provides that other modeling tools do not-… │
│ bc4f5d47c6ec │ Users need to be able to describe the infrastructure they want to build.\n       │
└──────────────┴──────────────────────────────────────────────────────────────────────────────────┘

In [6]:
# Filter rows
id_table = tables["id"]
id_table.filter(id_table["alias"].notnull())

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ entity_id                                    ┃ component_index ┃ component_type ┃ value                                        ┃ key                                          ┃ path                                                                             ┃ hash         ┃ alias                                        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ string                                       │ int64           │ string         │ string                                       │ string                                       │ string                                                                           │ string       │ string                                       │
├──────────────────────────────────────────────┼─────────────────┼────────────────┼──────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────────────────────────────────────────────────────────────────────┼──────────────┼──────────────────────────────────────────────┤
│ ecs_framework_base_requirement               │               0 │ id             │ ecs_framework_base_requirement               │ describe_infrastructure_with_ecs_framework   │ iacs.be_a_powerful_tool_for_solutions_architecture.provide_quality_end_to_end_d… │ 457ae1cf454a │ ecs_framework_base_requirement               │
│ ecs_framework_requirements                   │               0 │ id             │ ecs_framework_requirements                   │ requirements                                 │ ecs_framework.requirements                                                       │ ccc10301a979 │ ecs_framework_requirements                   │
│ describe_hierarchical_requirements           │               0 │ id             │ describe_hierarchical_requirements           │ describe_hierarchical_requirements           │ ecs_framework.requirements.describe_requirements.describe_hierarchical_requirem… │ dc2741f2714b │ describe_hierarchical_requirements           │
│ enable_multiple_inheritance_for_requirements │               0 │ id             │ enable_multiple_inheritance_for_requirements │ enable_multiple_inheritance_for_requirements │ ecs_framework.requirements.describe_requirements.describe_hierarchical_requirem… │ 09eeb1f3193d │ enable_multiple_inheritance_for_requirements │
│ describe_hierarchical_systems                │               0 │ id             │ describe_hierarchical_systems                │ describe_hierarchical_systems                │ ecs_framework.requirements.describe_systems.describe_hierarchical_systems        │ 97c8e4981b15 │ describe_hierarchical_systems                │
│ ecs_hierarchical_requirement                 │               0 │ id             │ ecs_hierarchical_requirement                 │ describe_hierarchical_structure              │ ecs_framework.requirements.describe_hierarchical_structure                       │ 153938a9792b │ ecs_hierarchical_requirement                 │
│ tag                                          │               0 │ id             │ tag                                          │ tag                                          │ component.tag                                                                    │ 76110bb3a835 │ tag                                          │
│ value                                        │               0 │ id             │ value                                   

## 4. Joins

In [7]:
# Join id and description by entity_id
joined = id_table.join(
    description,
    id_table.entity_id == description.entity_id,
).select(
    path=id_table.path,
    key=id_table.key,
    description=description.value,
)
joined.limit(20)

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ path                                                                             ┃ key                                                        ┃ description                                                                      ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ string                                                                           │ string                                                     │ string                                                                           │
├──────────────────────────────────────────────────────────────────────────────────┼────────────────────────────────────────────────────────────┼──────────────────────────────────────────────────────────────────────────────────┤
│ file_info                                                                        │ file_info                                                  │ This file contains the key entities that define and describe how iacs works.\n   │
│ iacs.be_a_powerful_tool_for_solutions_architecture                               │ be_a_powerful_tool_for_solutions_architecture              │ iacs (short for Infrastructure-as-Code Sketch) is software designed to help tec… │
│ iacs.be_a_powerful_tool_for_solutions_architecture.provide_quality_end_to_end_d… │ provide_quality_end_to_end_documentation_of_infrastructure │ Provide a means to comprehensively and accurately document infrastructure.\n     │
│ iacs.be_a_powerful_tool_for_solutions_architecture.provide_quality_end_to_end_d… │ parse_infrastructure_in_a_wide_variety_of_formats          │ If infrastructure is defined elsewhere in an existing format, we don't want to … │
│ iacs.be_a_powerful_tool_for_solutions_architecture.provide_quality_end_to_end_d… │ parse_a_variety_of_IaC_formats                             │ We don't want to duplicate other IaC code if possible.\n                         │
│ iacs.be_a_powerful_tool_for_solutions_architecture.provide_quality_end_to_end_d… │ parse_code_languages                                       │ Code such as Python also provides clear definitions.\n                           │
│ iacs.be_a_powerful_tool_for_solutions_architecture.provide_quality_end_to_end_d… │ compatible_with_version_control                            │ Version control is essential for reliability.\n                                  │
│ iacs.be_a_powerful_tool_for_solutions_architecture.provide_quality_end_to_end_d… │ aggregate_all_definitions_in_one_location                  │ A source of truth is not *a* source of truth if it is distributed.\n             │
│ iacs.be_a_powerful_tool_for_solutions_architecture.provide_quality_end_to_end_d… │ comprehensively_document_infrastructure                    │ This is one of the key features iacs provides that other modeling tools do not-… │
│ iacs.be_a_powerful_tool_for_solutions_architecture.provide_quality_end_to_end_d… │ accept_user_defined_infrastructure                         │ Users need to be able to describe the infrastructure they want to build.\n       │
│ …                                                                                │ …                                                          │ …                                                                                │
└──────────────────────────────────────────────────────────────────────────────────┴────────────────────────────────────────────────────────────┴──────────────────────────────────────────────────────────────────────────────────┘

## 5. Aggregation and group_by

In [8]:
# Count entities per component table
for name, tbl in tables.items():
    n = tbl.entity_id.nunique().execute()
    print(f"{name}: {n} entities")

description: 122 entities
file_info: 1 entities
id: 144 entities
input: 8 entities
output: 7 entities
parent: 141 entities
system: 4 entities
todo: 7 entities


In [9]:
# Group by with aggregation — e.g., entities with multiple descriptions
description.group_by("entity_id").aggregate(
    n_descriptions=description.value.count()
).filter(ibis._.n_descriptions > 1)

┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┓
┃ entity_id   ┃ n_descriptions ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━┩
│ string      │ int64          │
├─────────────┼────────────────┤
│ categorical │              2 │
└─────────────┴────────────────┘

## 6. Mutate — add computed columns

In [10]:
# Add a column that shows description length
description.mutate(
    desc_length=description.value.length()
).order_by(ibis.desc("desc_length")).limit(10)

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ entity_id                          ┃ component_index ┃ component_type ┃ value                                                                            ┃ desc_length ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ string                             │ int64           │ string         │ string                                                                           │ int32       │
├────────────────────────────────────┼─────────────────┼────────────────┼──────────────────────────────────────────────────────────────────────────────────┼─────────────┤
│ 0634a10a4a0f                       │               2 │ description    │ A component is a dictionary of data belonging to one or more entities. Componen… │        1095 │
│ 6c837dbd9816                       │               1 │ description    │ iacs stores and manipulates infrastructure data following an entity-component-s… │         376 │
│ 77eee9b3533b                       │               2 │ description    │ To ensure IDs are both unique and readable, a powerful solution is to define en… │         306 │
│ 331d6bf7c846                       │               2 │ description    │ First step of the process is to identify the big picture requirements of the sy… │         249 │
│ 085b2f4e818d                       │               2 │ description    │ iacs (short for Infrastructure-as-Code Sketch) is software designed to help tec… │         248 │
│ 6d8e20ddd998                       │               2 │ description    │ The entity-centered ECS format organizes the data around entities, grouping tog… │         236 │
│ be86805fbdc7                       │               2 │ description    │ One of the core parts of an ECS framework. Whereas entities and components reco… │         229 │
│ describe_hierarchical_requirements │               2 │ description    │ Our specific approach to requirements is to start with the fundamental requirem… │         227 │
│ 7d6595050a45                       │               2 │ description    │ The infrastructure that supports the ECS framework. The system consists of a Re… │         216 │
│ dff9384c5ada                       │               2 │ description    │ When the ((component_type)) has ((relation)) as an ancestor and the ((component… │         206 │
└────────────────────────────────────┴─────────────────┴────────────────┴──────────────────────────────────────────────────────────────────────────────────┴─────────────┘

## 7. Method chaining

Ibis expressions compose naturally. Use `ibis._` to refer to the
current table in a chain.

In [11]:
# Chain: join, filter, select, order
(
    id_table
    .join(description, "entity_id")
    .mutate(desc_length=description.value.length())
    .filter(ibis._.desc_length > 20)
    .select("path", "value", "desc_length")
    .order_by(ibis.desc("desc_length"))
    .limit(10)
)

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ path                                                                             ┃ value                              ┃ desc_length ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ string                                                                           │ string                             │ int32       │
├──────────────────────────────────────────────────────────────────────────────────┼────────────────────────────────────┼─────────────┤
│ ecs_framework.core_concepts.component                                            │ 0634a10a4a0f                       │        1095 │
│ ecs_framework                                                                    │ 6c837dbd9816                       │         376 │
│ ecs_framework.requirements.describe_hierarchical_structure.define_entity_ids_by… │ 77eee9b3533b                       │         306 │
│ iacs.be_a_powerful_tool_for_solutions_architecture.help_improve_the_quality_of_… │ 331d6bf7c846                       │         249 │
│ iacs.be_a_powerful_tool_for_solutions_architecture                               │ 085b2f4e818d                       │         248 │
│ ecs_framework.ecs_data_formats.entity_centered                                   │ 6d8e20ddd998                       │         236 │
│ ecs_framework.core_concepts.system                                               │ be86805fbdc7                       │         229 │
│ ecs_framework.requirements.describe_requirements.describe_hierarchical_requirem… │ describe_hierarchical_requirements │         227 │
│ ecs_framework.ecs_system                                                         │ 7d6595050a45                       │         216 │
│ ecs_framework.ecs_data_formats.entity_centered.component_item.dictionary_compon… │ dff9384c5ada                       │         206 │
└──────────────────────────────────────────────────────────────────────────────────┴────────────────────────────────────┴─────────────┘

## 8. SQL escape hatch

You can always drop down to raw SQL when needed.

In [12]:
con.sql("SELECT * FROM description LIMIT 5")

┏━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ entity_id    ┃ component_index ┃ component_type ┃ value                                                                            ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ string       │ int64           │ string         │ string                                                                           │
├──────────────┼─────────────────┼────────────────┼──────────────────────────────────────────────────────────────────────────────────┤
│ 080505fba0c9 │               1 │ description    │ This file contains the key entities that define and describe how iacs works.\n   │
│ 085b2f4e818d │               2 │ description    │ iacs (short for Infrastructure-as-Code Sketch) is software designed to help tec… │
│ 847469f00f2d │               2 │ description    │ Provide a means to comprehensively and accurately document infrastructure.\n     │
│ 5cdb11e7f0af │               2 │ description    │ If infrastructure is defined elsewhere in an existing format, we don't want to … │
│ aa4b968e2d6d │               2 │ description    │ We don't want to duplicate other IaC code if possible.\n                         │
└──────────────┴─────────────────┴────────────────┴──────────────────────────────────────────────────────────────────────────────────┘

## 9. Convert back to pandas

In [13]:
# Any Ibis expression can be converted to a pandas DataFrame
result_df = description.select("entity_id", "value").to_pandas()
result_df.head()

,entity_id,value
0,080505fba0c9,This file contains the key entities that defin...
1,085b2f4e818d,iacs (short for Infrastructure-as-Code Sketch)...
2,847469f00f2d,Provide a means to comprehensively and accurat...
3,5cdb11e7f0af,If infrastructure is defined elsewhere in an e...
4,aa4b968e2d6d,We don't want to duplicate other IaC code if p...


## 10. Inspect the compiled SQL

Use `ibis.to_sql()` to see what SQL Ibis generates.

In [14]:
expr = (
    id_table
    .join(description, "entity_id")
    .select("path", "value")
    .limit(10)
)
print(ibis.to_sql(expr))

SELECT
  "t2"."path",
  "t2"."value"
FROM "id" AS "t2"
INNER JOIN "description" AS "t3"
  ON "t2"."entity_id" = "t3"."entity_id"
LIMIT 10


## 11. Try your own queries

Available tables: `tables.keys()` shows all registered component tables.
Use `.execute()` or `.to_pandas()` to materialize results.

In [15]:
ibis.dtype("json")

JSON(binary=False, nullable=True)